# 00 — Shared environment and four-arm setup

**Purpose:** check one Python environment for the whole evaluation and make the two models × two NOUL designs visible from the start. Binary JEV-9B, binary JEV-27B v1.1, hybrid JEV-9B and hybrid JEV-27B v1.1 share preparation and reference data; questions, model identities and result directories remain separate.

| Item | Description |
| --- | --- |
| Inputs | Existing uv environment, project/lock files and YAML settings |
| Outputs | Environment, reproducibility and four-arm configuration tables in memory |
| Writes / API calls | None; no package installation, data conversion or inference in this notebook |
| Main experiment sources | `configs/evaluations/comparison.yaml`, independent of the generic demo default |

Notebook 00 can check configuration without private datasets/results. Missing artifact availability is reported rather than fabricated. Follow 01 for shared-data validation, 02 for mock connection examples, 03 for saved-run reuse and 04–05 for the four-arm comparison. Use the project's `.venv` kernel and clear outputs before committing.

## 1. Open the shared project environment

In a terminal at the repository root, run `uv sync --locked --group notebooks`, then `uv run --locked --group notebooks jupyter lab`. These commands install the recorded packages and start Jupyter; this notebook does neither. The same environment serves every arm, so no separate binary/hybrid installation is needed.

Select the Python interpreter in this repository's `.venv`. A kernel executes notebook cells; restart it after code, configuration or dependency changes and rerun prerequisites. The generic initialization below uses synthetic settings to check the environment only; it does not select one experiment instead of another.

In [ ]:
try:
    from jev_eval.environment import initialize_notebook
except ModuleNotFoundError:
    raise RuntimeError(
        "Select this project's .venv Python kernel. From the repository terminal, "
        "run: uv sync --locked --group notebooks"
    ) from None

project = initialize_notebook()

## 2. Check Python and dependencies

The table below shows whether the project environment and packages are available. Missing hosted credentials are normal in demo mode. If the environment row is false, switch to the `.venv` kernel. If a package is missing, run the setup command above and restart the kernel.

In [ ]:
from jev_eval.environment import check_environment

check_environment(project)

## 3. Distinguish shared setup from experiment selection

YAML files contain readable settings. `configs/notebooks.yaml` controls the generic synthetic demonstration below; its `mode: demo` does **not** determine the four-arm analysis. `configs/evaluations/comparison.yaml` explicitly selects both binary and hybrid archives, each with both model workflows.

The table below describes the generic setup context, not observed JEV performance. Section 7 displays the actual four experiment configurations without opening private citations. Paths resolve from the repository root. Setup does not grant paid-call permission or change any workflow.

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {
            "evaluation": project.workflow.evaluation_config,
            "mode": project.workflow.execution.mode,
            "record_limit": project.workflow.execution.max_records,
            "logical_request_limit": project.workflow.execution.max_logical_requests,
            "paid_calls_enabled": project.workflow.execution.allow_paid_calls,
            "writes_enabled": project.workflow.output.allow_writes,
        }
    ]
)

## 4. Credentials are needed only for deliberate live requests

Saved-result inspection and mock examples need no API key. For a deliberate single-request live test, copy `.env.example` to `.env` only if it does not exist, and set `INFERENCE_API_KEY` locally. `.env` is ignored by Git; never print the key or the entire environment. Initialization does not overwrite variables already set outside `.env`. Restart the kernel after changing credentials.

Notebook 02's default demonstrations are mock-only. Original one-state/one-question and separate one-state/seven-question live-test safeguards remain available, with explicit flags off. Notebook 03 reuses existing archives; this setup notebook neither sends requests nor edits execution permissions.

## 5. Record reproducibility information

The lockfile fixes dependency resolutions. The Git revision identifies the code, while a dirty working tree means there are uncommitted changes. A missing commit is expected before the first commit. Final research runs should use a recorded, reviewed revision.

A fixed seed makes local sampling repeatable. It cannot guarantee identical responses from a hosted model that the provider may update.

In [ ]:
from jev_eval.environment import reproducibility_summary

reproducibility_summary(project)

## 6. What is shared and what remains separate

All arms use the same Python environment and, after validation in notebook 01, one citation cohort and one human-reference CSV. This prevents preprocessing differences or hidden label exposure from confounding the screening comparison. Model IDs, task definitions, request counts and output directories remain arm-specific.

A fixed seed supports repeatable local sampling; it does not guarantee identical future hosted responses. An app/version name is not a verified immutable weight revision. Historical inference fingerprints remain unchanged when analysis code is updated.

## 7. Review all four configurations without reading private data

This configuration-only table names binary and hybrid for both models. It shows paths, question counts, configured task fingerprints, existing-manifest availability and YAML permissions. Availability does not assert a complete or valid saved run; notebooks 03–05 perform archive checks. True YAML paid/write permissions do not mean this notebook will submit requests.

**The archived binary runs predate hybrid-v1 wording.** Their configured overall definitions can differ. Reviewed binary-v1 templates still exist for a future matched experiment, but are not silently substituted for the saved binary results. Report this distinction when explaining differences in performance.

R1 remains disabled pending protocol review. Formal tuning requires the original development/tuning membership and resolution of the 7,008/7,000 discrepancy. The planned 2,000 unseen studies are not loaded or used here.

In [ ]:
from jev_eval.comparison import comparison_setup_table
from jev_eval.screening import load_screening_settings

four_arm_setup = comparison_setup_table(project.root)
display(four_arm_setup)
screening_settings = load_screening_settings(project.root)
print("Four-arm source configuration: configs/evaluations/comparison.yaml")
print("R1:", screening_settings.r1_policy)
print(
    "Validation studies planned after colleague review:",
    screening_settings.planned_validation_records,
)
print("No dataset was opened, no API request was sent and no file was written.")